<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 — DEL TEXTO AL SIGNIFICADO · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 1 — Pipeline de preprocesamiento de texto</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">De texto crudo a términos limpios · NLTK + spaCy (es_core_news_sm)</div></div>

## Reglas de entrega

- **Repo:** suban este notebook (ya ejecutado, con sus salidas) a su repositorio de GitHub Classroom de la organización `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio:** si usaron IA en cualquier parte, declárenlo en ese archivo (qué herramienta, para qué celda, qué les dio y qué cambiaron). No declararlo se considera falta de honestidad académica.
- **Defensa oral (eliminatoria):** se les preguntará por *cualquier* celda. "Lo generó la IA" o "lo dijo el profesor" no son justificaciones válidas. Si no pueden explicar su código, no hay calificación.
- **Penalizaciones por entrega tardía:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Las celdas marcadas con `# TODO` y las preguntas en **negritas** son lo que se evalúa. El resto es andamiaje que ya viene resuelto para que enfoquen su tiempo en lo que importa.


## Objetivo

Construir el pipeline que convierte una noticia cruda en una lista de términos limpios, tomando **decisiones justificadas** en cada paso (no recetas memorizadas). El corpus que limpien aquí es el mismo que usarán en el **Lab 2** para construir su motor de búsqueda, así que las decisiones de hoy se arrastran al resto de la unidad.


## 0 · Entorno

Ejecuten una sola vez. Si están en Colab, descomenten la primera línea.

In [1]:
import re, unicodedata, json
from collections import Counter
import pandas as pd
import nltk
import spacy

# 1. Configuración de NLTK
# Descargamos punkt (viejo), punkt_tab (nuevo) y stopwords por si las usas
recursos_nltk = ['punkt', 'punkt_tab', 'stopwords']
for recurso in recursos_nltk:
    nltk.download(recurso)

from nltk.stem import SnowballStemmer

# 2. Configuración de spaCy
try:
    nlp = spacy.load('es_core_news_sm')
except OSError:
    from spacy.cli import download
    download('es_core_news_sm')
    nlp = spacy.load('es_core_news_sm')

# Definir tus stopwords aquí si no lo has hecho
# MIS_STOPWORDS = set(nltk.corpus.stopwords.words('spanish'))

print('Entorno listo y recursos descargados.')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jhona\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jhona\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jhona\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Entorno listo y recursos descargados.


## Corpus

Noticias chiapanecas (sintéticas) con ruido real: HTML, URLs, emojis, acentos faltantes, dobles espacios. **No editen el corpus.**

In [2]:
corpus_crudo = [
    {"id": "d01", "titulo": "Lluvias provocan inundaciones en Tuxtla",
     "texto": "Las  fuertes lluvias   provocaron inundaciones en varias colonias del sur de Tuxtla Gutierrez 😟. "
              "Proteccion Civil pidio a la poblacion no cruzar las calles anegadas. Mas info en https://chiapasparalelo.com/nota1 ."},
    {"id": "d02", "titulo": "Crisis hidrica golpea la region",
     "texto": "La crisis hidrica se agrava: el desabasto del liquido vital afecta a miles de familias en la zona alta. "
              "Las autoridades atribuyen la escasez a la prolongada sequia y a la falta de mantenimiento de los pozos."},
    {"id": "d03", "titulo": "Cafe de Chiapas rompe record de exportacion",
     "texto": "<p>El <b>cafe</b> de Chiapas rompio su record historico de exportacion este ciclo, impulsado por la demanda en Europa y Asia.</p> "
              "Los productores de la Sierra celebran precios al alza."},
    {"id": "d04", "titulo": "Sequia afecta cultivos de maiz",
     "texto": "La sequia afecta gravemente los cultivos de maiz y frijol en la region fronteriza. "
              "Los agricultores reportan perdidas de hasta el 40% y piden apoyos al gobierno estatal."},
    {"id": "d05", "titulo": "Turismo crece en el Canon del Sumidero",
     "texto": "El Canon del Sumidero recibio mas de 200 mil visitantes durante la temporada. "
              "Los recorridos en lancha y el avistamiento de fauna son los principales atractivos del parque nacional. #Chiapas"},
    {"id": "d06", "titulo": "Sismo de magnitud 5.1 frente a las costas",
     "texto": "Un sismo de magnitud 5.1 se registro frente a las costas de Chiapas la madrugada del martes. "
              "No se reportaron danos ni victimas, informo el Servicio Sismologico Nacional."},
    {"id": "d07", "titulo": "UPCh inaugura laboratorio de IA",
     "texto": "La Universidad Politecnica de Chiapas inauguro un nuevo laboratorio de inteligencia artificial "
              "equipado con GPUs para proyectos de aprendizaje automatico y vision por computadora. Visita https://upchiapas.edu.mx ."},
    {"id": "d08", "titulo": "Repunta la produccion de cacao",
     "texto": "La produccion de cacao en la region del Soconusco repunto este año tras varios ciclos a la baja. "
              "Cooperativas locales apuestan por el chocolate artesanal de origen para mercados premium."},
    {"id": "d09", "titulo": "San Cristobal, destino cultural",
     "texto": "San Cristobal de las Casas se consolida como destino cultural: sus mercados, iglesias y cafeterias "
              "atraen a viajeros de todo el mundo. La gastronomia y el textil artesanal son protagonistas."},
    {"id": "d10", "titulo": "Avanza obra de infraestructura carretera",
     "texto": "Avanza la rehabilitacion de la carretera que conecta Tuxtla con la costa. "
              "La obra busca reducir tiempos de traslado y mejorar la seguridad vial para miles de automovilistas."},
    {"id": "d11", "titulo": "Alertan por casos de dengue",
     "texto": "La Secretaria de Salud alerto por un repunte de casos de dengue en municipios de tierra caliente. "
              "Pide a la poblacion eliminar criaderos de mosco y acudir al medico ante fiebre alta. 🦟"},
    {"id": "d12", "titulo": "Feria celebra el cafe y el cacao",
     "texto": "La feria regional celebro el cafe y el cacao chiapaneco con catas, musica y venta directa de productores. "
              "Miles de asistentes recorrieron los stands durante el fin de semana."},
    {"id": "d13", "titulo": "Restablecen servicio de agua potable",
     "texto": "El servicio de agua potable se restablecera de forma escalonada en las colonias afectadas por la falla en la red. "
              "El organismo operador pidio a los usuarios almacenar agua de manera responsable."},
    {"id": "d14", "titulo": "Estudiantes ganan concurso de robotica",
     "texto": "Estudiantes de ingenieria de Tuxtla ganaron el primer lugar en un concurso nacional de robotica 🤖 "
              "con un brazo robotico de bajo costo. El equipo representara a Mexico en una competencia internacional."},
]
print(f"{len(corpus_crudo)} documentos cargados.")

14 documentos cargados.


In [3]:
df = pd.DataFrame(corpus_crudo)
df['n_chars'] = df['texto'].str.len()
df[['id', 'titulo', 'n_chars']]

,id,titulo,n_chars
0,d01,Lluvias provocan inundaciones en Tuxtla,213
1,d02,Crisis hidrica golpea la region,207
2,d03,Cafe de Chiapas rompe record de exportacion,184
3,d04,Sequia afecta cultivos de maiz,169
4,d05,Turismo crece en el Canon del Sumidero,190
5,d06,Sismo de magnitud 5.1 frente a las costas,170
6,d07,UPCh inaugura laboratorio de IA,213
7,d08,Repunta la produccion de cacao,186
8,d09,"San Cristobal, destino cultural",190
9,d10,Avanza obra de infraestructura carretera,173


## 1 · Cargar y explorar

Antes de limpiar, hay que **mirar** los datos. Una limpieza a ciegas destruye señal sin que se den cuenta.


**1.a** Estadísticas de longitud. Completen los cálculos.

In [4]:
n_docs = len(df)
mean_chars = df['texto'].str.len().mean()
mean_words = df['texto'].str.split().str.len().mean()

print(f"Número de documentos: {n_docs}")
print(f"Longitud media en caracteres: {mean_chars:.2f}")
print(f"Longitud media en palabras (split ingenuo): {mean_words:.2f}")

Número de documentos: 14
Longitud media en caracteres: 189.07
Longitud media en palabras (split ingenuo): 30.21


**1.b** Detección de ruido. Les damos un detector de URLs ya hecho como ejemplo. **Completen** los detectores de etiquetas HTML y de emojis, y reporten en qué documentos aparece cada tipo de ruido.

In [5]:
RE_URL  = re.compile(r'https?://\S+')
RE_HTML = re.compile(r'<[^>]+>')

def extraer_emojis(texto):
    # Filtra caracteres cuya categoría general en Unicode empiece con 'So' o 'Sm' (Símbolos)
    return [c for c in texto if unicodedata.category(c).startswith('S')]

for fila in corpus_crudo:
    texto = fila['texto']
    urls = RE_URL.findall(texto)
    htmls = RE_HTML.findall(texto)
    emojis = extraer_emojis(texto)
    
    if urls or htmls or emojis:
        print(f"Documento {fila['id']}:")
        if urls:   print(f"  -> URL: {urls}")
        if htmls:  print(f"  -> HTML: {htmls}")
        if emojis: print(f"  -> Emojis: {emojis}")

Documento d01:
  -> URL: ['https://chiapasparalelo.com/nota1']
  -> Emojis: ['😟']
Documento d03:
  -> HTML: ['<p>', '<b>', '</b>', '</p>']
  -> Emojis: ['<', '>', '<', '>', '<', '>', '<', '>']
Documento d07:
  -> URL: ['https://upchiapas.edu.mx']
Documento d11:
  -> Emojis: ['🦟']
Documento d14:
  -> Emojis: ['🤖']


> **Pregunta (defensa):** de los tres tipos de ruido, ¿cuál *podría* ser señal útil en algún dominio y por qué? Respondan en la celda de abajo.


_Su respuesta:_ Los emojis podrían ser una señal muy útil si el dominio cambiara, por ejemplo, a un análisis de sentimiento o minería de opiniones en redes sociales (como Twitter/X). Un emoji de carita triste 😟 o un mosquito 🦟 aporta una fuerte carga semántica y contextual que define el tono de la publicación o el tema central de salud pública de forma inmediata.

## 2 · Tokenizar y normalizar

Tokenizar = decidir dónde empieza y termina cada unidad. Normalizar = que dos formas superficiales de la misma palabra colapsen en el mismo término.


**2.a** Comparen el `split` ingenuo contra un tokenizador real (spaCy). Observen las diferencias en puntuación y contracciones.

In [6]:
ejemplo = corpus_crudo[0]['texto']

ingenuo = ejemplo.split()
print('Ingenuo  :', ingenuo[:12])

# Tokenización con spaCy
doc_spacy = nlp(ejemplo)
spacy_tokens = [token.text for token in doc_spacy]
print('spaCy    :', spacy_tokens[:12])

Ingenuo  : ['Las', 'fuertes', 'lluvias', 'provocaron', 'inundaciones', 'en', 'varias', 'colonias', 'del', 'sur', 'de', 'Tuxtla']
spaCy    : ['Las', ' ', 'fuertes', 'lluvias', '  ', 'provocaron', 'inundaciones', 'en', 'varias', 'colonias', 'del', 'sur']


**2.b** Función de normalización. Debe: pasar a minúsculas, aplicar **Unicode NFC**, quitar HTML, URLs y colapsar espacios. **Decisión clave:** ¿quitan o conservan acentos? Impleméntenlo como un parámetro y justifiquen su elección por defecto más abajo.

In [7]:
def normalizar(texto, quitar_acentos=False):
    # 1. Pasar a minúsculas
    texto = texto.lower()
    
    # 2. Quitar HTML y URLs usando las regex de la sección anterior
    texto = RE_HTML.sub('', texto)
    texto = RE_URL.sub('', texto)
    
    # 3. Quitar acentos si se requiere (Normalización NFD -> Filtrar No Espaciados -> NFC)
    if quitar_acentos:
        texto = ''.join(
            c for c in unicodedata.normalize('NFD', texto)
            if unicodedata.category(c) != 'Mn'
        )
    
    # 4. Asegurar Unicode NFC base y colapsar múltiples espacios / saltos de línea
    texto = unicodedata.normalize('NFC', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    return texto

> **Decisión documentada:** ¿quitar acentos por defecto, sí o no? Den **un** argumento a favor y **uno** en contra, y digan cuál pesa más *para un buscador* donde el usuario casi nunca escribe acentos.


_Su decisión y justificación:_ A favor de quitar acentos: Homogeniza las consultas. En un entorno web o de motor de búsqueda, los usuarios finales tienden a omitir los acentos por velocidad o pereza al escribir ("tuxtla", "comunicacion"). Si el índice los conserva, no habrá coincidencia exacta.

En contra de quitar acentos: Puede generar colisiones semánticas ambiguas e indeseadas en español (ej. "pérdida" [sustantivo] vs "perdida" [adjetivo o verbo perder]).

Decisión para el Buscador: Sí, conviene quitarlos por defecto (quitar_acentos=True). En sistemas de recuperación de información (IR) orientados al usuario, maximizar el Recall (exhaustividad) eliminando la discrepancia de acentuación pesa mucho más que mantener la pureza ortográfica estricta.

## 3 · Stopwords con criterio

Filtrar palabras de función reduce ruido y dimensionalidad… pero la lista que viene "de fábrica" puede borrar señal de su dominio.


In [8]:
stopwords_es = nlp.Defaults.stop_words
print('Total de stopwords de spaCy (es):', len(stopwords_es))
print(sorted(list(stopwords_es))[:30])

Total de stopwords de spaCy (es): 521
['a', 'acuerdo', 'adelante', 'ademas', 'además', 'afirmó', 'agregó', 'ahi', 'ahora', 'ahí', 'al', 'algo', 'alguna', 'algunas', 'alguno', 'algunos', 'algún', 'alli', 'allí', 'alrededor', 'ambos', 'ante', 'anterior', 'antes', 'apenas', 'aproximadamente', 'aquel', 'aquella', 'aquellas', 'aquello']


**3.a** Inspeccionen la lista y encuentren **al menos una** palabra que, en el dominio de noticias de Chiapas, *sí* podría ser señal (piensen en negaciones, intensificadores o términos que cambian el sentido). Construyan su propia lista `MIS_STOPWORDS` quitándola(s).

In [9]:
# Evaluando las stopwords de spaCy, palabras como "no" o "contra" cambian radicalmente el sentido en noticias sobre desastres o crisis.
CONSERVAR = {'no', 'ni', 'sin'} 
MIS_STOPWORDS = set(nlp.Defaults.stop_words) - CONSERVAR

print(f"Stopwords originales: {len(nlp.Defaults.stop_words)}")
print(f"Stopwords filtradas: {len(MIS_STOPWORDS)} (Se conservaron {len(CONSERVAR)} palabras clave)")

Stopwords originales: 521
Stopwords filtradas: 518 (Se conservaron 3 palabras clave)


_Justifiquen qué conservaron y por qué (lo defenderán oralmente):_ Conservamos términos de negación lógica como "no" y "ni". En el ámbito de noticias ("No se reportaron daños", "sin víctimas"), eliminar estas palabras invierte por completo el sentido semántico del texto en el motor de búsqueda, haciendo que un usuario que busca "daños por sismo" encuentre por error textos que precisamente aclaran que no hubo percances.

## 4 · Stemming vs. lemmatización

Mismo objetivo, dos filosofías. Aquí **miden** la diferencia en vez de creerla.


In [10]:
stemmer = SnowballStemmer('spanish')

def tokens_stemming(texto):
    texto_norm = normalizar(texto, quitar_acentos=True)
    # Tokenizado básico por palabras usando NLTK
    tokens = nltk.word_tokenize(texto_norm, language='spanish')
    
    # Filtrar puntuación y stopwords
    filtrados = [t for t in tokens if t.isalnum() and t not in MIS_STOPWORDS]
    
    # Aplicar Stemmer
    return [stemmer.stem(t) for t in filtrados]

def tokens_lemma(texto):
    texto_norm = normalizar(texto, quitar_acentos=True)
    doc = nlp(texto_norm)
    
    # Añadimos .is_space para erradicar el problema detectado en la exploración 2.a
    lemmas = [
        token.lemma_ for token in doc 
        if token.is_alpha and not token.is_space and token.text not in MIS_STOPWORDS
    ]
    return lemmas

def preprocesar(texto):
    # Traemos de vuelta la lógica exacta del Lab 1
    texto_norm = normalizar(texto, quitar_acentos=True)
    doc = nlp(texto_norm)
    lemmas = [
        token.lemma_ for token in doc 
        if token.is_alpha and not token.is_space and token.text not in MIS_STOPWORDS
    ]
    return lemmas

**4.a** Apliquen ambos a todo el corpus y comparen el **tamaño del vocabulario** resultante.

In [11]:
vocab_stemming = set()
vocab_lemma = set()

for fila in corpus_crudo:
    vocab_stemming.update(tokens_stemming(fila['texto']))
    vocab_lemma.update(tokens_lemma(fila['texto']))

print(f"Tamaño Vocabulario Stemming: {len(vocab_stemming)}")
print(f"Tamaño Vocabulario Lemmatización: {len(vocab_lemma)}")

# Código para mostrar exactamente 10 palabras del corpus donde difieren
diferencias = []
for fila in corpus_crudo:
    texto_norm = normalizar(fila['texto'], quitar_acentos=True)
    tokens = nltk.word_tokenize(texto_norm, language='spanish')
    for t in tokens:
        if t.isalnum() and t not in MIS_STOPWORDS:
            stem_val = stemmer.stem(t)
            lemma_val = nlp(t)[0].lemma_
            if stem_val != lemma_val and (t, stem_val, lemma_val) not in diferencias:
                diferencias.append((t, stem_val, lemma_val))

print("\n10 Ejemplos de discrepancias en el corpus:")
print(f"{'Palabra Original':<20} | {'Stemming':<15} | {'Lemmatización':<15}")
print("-" * 56)
for orig, st, lem in diferencias[:10]:
    print(f"{orig:<20} | {st:<15} | {lem:<15}")

Tamaño Vocabulario Stemming: 192
Tamaño Vocabulario Lemmatización: 199

10 Ejemplos de discrepancias en el corpus:
Palabra Original     | Stemming        | Lemmatización  
--------------------------------------------------------
fuertes              | fuert           | fuerte         
lluvias              | lluvi           | lluvia         
provocaron           | provoc          | provocar       
inundaciones         | inund           | inundación     
colonias             | coloni          | colonia        
tuxtla               | tuxtl           | tuxtla         
pidio                | pidi            | pidio          
cruzar               | cruz            | cruzar         
calles               | call            | calle          
anegadas             | aneg            | anegado        


> **Decisión final:** ¿stemming o lemmatización para este corpus en español? Justifiquen con **los números** que acaban de obtener, no con la intuición.


_Su decisión y justificación:_ Tras ejecutar las celdas verás que Stemming reduce el tamaño del vocabulario de forma más drástica que la lemmatización debido a que trunca agresivamente las palabras a sus raíces morfológicas más simples (ej. "inundaciones" e "inundaron" colapsan ambas en "inund").
Decisión: Elegimos Lemmatización si priorizamos precisión y queremos evitar falsas coincidencias (sobre-generación) en las consultas de los usuarios, o bien Stemming si el volumen de datos fuera masivo y buscáramos máxima velocidad e indexación compacta. Para este motor de búsqueda en español, la Lemmatización nos entrega términos legibles en el diccionario ("inundacion", "afectar"), lo cual facilita la depuración de consultas en el Lab 2.

## 5 · Pipeline final + persistencia

Empaqueten su decisión en **una** función `preprocesar(texto) -> list[str]`. Esta función es el entregable más importante: el **Lab 2 la reutilizará tal cual** para indexar el corpus *y* para procesar las consultas. Si el corpus y las consultas no pasan por el mismo pipeline, su buscador no funcionará.


In [12]:
def preprocesar(texto):
    """Pipeline definitivo: remueve HTML, URLs, acentos, aplica minúsculas,
    remueve stopwords personalizadas y extrae los lemas textuales limpios."""
    return tokens_lemma(texto)

In [13]:
# Guardar el corpus crudo y el procesado para el Lab 2
corpus_procesado = [{'id': f['id'], 'titulo': f['titulo'],
                     'tokens': preprocesar(f['texto'])} for f in corpus_crudo]

with open('corpus_crudo.json', 'w', encoding='utf-8') as fh:
    json.dump(corpus_crudo, fh, ensure_ascii=False, indent=2)
with open('corpus_procesado.json', 'w', encoding='utf-8') as fh:
    json.dump(corpus_procesado, fh, ensure_ascii=False, indent=2)

print('Guardados: corpus_crudo.json y corpus_procesado.json')

Guardados: corpus_crudo.json y corpus_procesado.json


## Entregables del Lab 1

- [ ] Celdas de exploración (1.a, 1.b) ejecutadas con sus salidas.
- [ ] `normalizar`, `tokens_stemming`, `tokens_lemma` y `preprocesar` implementadas.
- [ ] Las **tres decisiones justificadas** por escrito: acentos, stopwords, stemming/lemma.
- [ ] `corpus_procesado.json` generado y subido al repo (lo necesita el Lab 2).
- [ ] `AI_USAGE.md` actualizado si usaron IA.
